# Train Hate Speech Detection on Colab

This notebook clones or updates the project from GitHub, prepares data, then calls the production Python modules in `src/`.

Default branch is `main`. Change `BRANCH` in the first code cell if you want to train from another branch, for example `refactor`.

In [1]:
from pathlib import Path
import os
import subprocess

REPO_URL = "https://github.com/thong7d/hate-speech-detection.git"
BRANCH = "thong"
PROJECT_DIR = Path("/content/hate-speech-detection")

if PROJECT_DIR.exists():
    if not (PROJECT_DIR / ".git").exists():
        raise RuntimeError(f"{PROJECT_DIR} exists but is not a git repository. Remove it or choose another PROJECT_DIR.")
    os.chdir(PROJECT_DIR)
    subprocess.run(["git", "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "checkout", BRANCH], check=True)
    subprocess.run(["git", "pull", "--ff-only", "origin", BRANCH], check=True)
else:
    subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, str(PROJECT_DIR)], check=True)
    os.chdir(PROJECT_DIR)

print("Project directory:", Path.cwd())
subprocess.run(["git", "branch", "--show-current"], check=True)

Project directory: /content/hate-speech-detection


CompletedProcess(args=['git', 'branch', '--show-current'], returncode=0)

In [2]:
from pathlib import Path

skip_packages = ("underthesea", "gradio", "pyngrok", "google-generativeai")
source = Path("requirements.txt")
target = Path("requirements-colab.txt")
lines = []
for line in source.read_text(encoding="utf-8").splitlines():
    stripped = line.strip()
    if stripped and not stripped.startswith(skip_packages):
        lines.append(line)
target.write_text("\n".join(lines) + "\n", encoding="utf-8")
print(target.read_text(encoding="utf-8"))

transformers==4.40.2
datasets==2.19.2
torch>=2.1.0
accelerate==0.29.3
scikit-learn==1.4.2
pandas==2.2.2
pyarrow==16.0.0
requests==2.32.2
PyYAML==6.0.2
matplotlib==3.9.0
seaborn==0.13.2
langdetect==1.0.9
fastapi==0.111.0
uvicorn==0.29.0
omegaconf==2.3.0
sentencepiece==0.2.0
protobuf==4.25.3



In [3]:
!python -m pip install -U pip setuptools wheel
# Gỡ bỏ TensorFlow triệt để để ngăn transformers gọi các module gây lỗi protobuf
!python -m pip uninstall -y tensorflow
# Không can thiệp vào protobuf, để Colab dùng bản mặc định
!python -m pip install -r requirements-colab.txt
!python -m pip install huggingface_hub sentencepiece
!python -m pip uninstall -y peft

Found existing installation: tensorflow 2.20.0
Uninstalling tensorflow-2.20.0:
  Successfully uninstalled tensorflow-2.20.0
  Using cached protobuf-4.25.3-cp37-abi3-manylinux2014_x86_64.whl.metadata (541 bytes)
Using cached protobuf-4.25.3-cp37-abi3-manylinux2014_x86_64.whl (294 kB)
  Attempting uninstall: protobuf
    Found existing installation: protobuf 3.20.3
    Uninstalling protobuf-3.20.3:
      Successfully uninstalled protobuf-3.20.3
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires tensorflow>=2.2.0, which is not installed.
grain 0.2.16 requires protobuf>=5.28.3, but you have protobuf 4.25.3 which is incompatible.
google-cloud-pubsub 2.37.0 requires protobuf<8.0.0,>=4.25.8, but you have protobuf 4.25.3 which is incompatible.
google-cloud-bigquery-storage 2.37.0 requires protobuf<8.0.0,>=4.25.8, but you have protobuf 4.25.3 whi

In [4]:
from pathlib import Path

from src.data.download import download_and_extract
from src.data.preprocessing import process_split

raw_paths = {
    "train": Path("data/raw/vihsd/train.csv"),
    "dev": Path("data/raw/vihsd/dev.csv"),
    "test": Path("data/raw/vihsd/test.csv"),
}
processed_paths = {
    "train": Path("data/processed/train.parquet"),
    "dev": Path("data/processed/dev.parquet"),
    "test": Path("data/processed/test.parquet"),
}

if not all(path.exists() for path in raw_paths.values()):
    download_and_extract("configs/paths.yaml")

for split in ("train", "dev", "test"):
    if not processed_paths[split].exists():
        df = process_split(raw_paths[split], processed_paths[split])
        print(f"{split}: {len(df)} rows -> {processed_paths[split]}")
    else:
        print(f"{split}: found {processed_paths[split]}")

train: found data/processed/train.parquet
dev: found data/processed/dev.parquet
test: found data/processed/test.parquet


In [5]:
import os

try:
    from google.colab import userdata
    token = userdata.get("HF_TOKEN")
    if token:
        os.environ["HF_TOKEN"] = token
except Exception:
    pass

if os.environ.get("HF_TOKEN"):
    from huggingface_hub import login
    login(token=os.environ["HF_TOKEN"])
else:
    print("Khong du du lieu de xac minh HF_TOKEN hoac HF_TOKEN chua duoc cau hinh")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [6]:
!python -c "import torch, transformers, pandas, pyarrow, sklearn, src; print('OK')"

OK


In [7]:
!python -m src.training.train --config configs/train.yaml

[TRAIN] Final training set size: 56648 samples
[TRAIN] Validation set size: 2639 samples
[TRAIN] Label distribution: {2: 20470, 0: 19970, 1: 16208}
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
config.json: 100% 678/678 [00:00<00:00, 4.01MB/s]
vocab.txt: 895kB [00:00, 9.22MB/s]
bpe.codes: 1.14MB [00:00, 11.0MB/s]
tokenizer.json: 3.13MB [00:00, 16.6MB/s]
pytorch_model.bin: 100% 540M/540M [00:03<00:00, 141MB/s]
Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at vinai/phobert-base-v2 and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predict

In [8]:
!python -m src.evaluation.evaluate --config configs/train.yaml

[EVAL] Error analysis written: 937 errors -> /content/hate-speech-detection/results/error_analysis.csv
{'accuracy': 0.8583, 'macro_f1': 0.6482, 'weighted_f1': 0.8593, 'precision_CLEAN': 0.9299, 'recall_CLEAN': 0.9267, 'f1_CLEAN': 0.9283, 'precision_OFFENSIVE': 0.4095, 'recall_OFFENSIVE': 0.4482, 'f1_OFFENSIVE': 0.428, 'precision_HATE': 0.5985, 'recall_HATE': 0.5785, 'f1_HATE': 0.5883, 'critical_f1': 0.5081, 'critical_recall': 0.5133, 'offensive_priority_f1': 0.4761, 'balanced_critical_f1': 0.5362, 'confusion_matrix': [[5079, 206, 196], [174, 199, 71], [209, 81, 398]], 'classification_report': {'CLEAN': {'precision': 0.929879165140974, 'recall': 0.926655719759168, 'f1-score': 0.9282646440646989, 'support': 5481.0}, 'OFFENSIVE': {'precision': 0.4094650205761317, 'recall': 0.44819819819819817, 'f1-score': 0.42795698924731185, 'support': 444.0}, 'HATE': {'precision': 0.5984962406015037, 'recall': 0.5784883720930233, 'f1-score': 0.5883222468588323, 'support': 688.0}, 'accuracy': 0.858309390

In [9]:
push_flag = "--push-to-hub" if os.environ.get("HF_TOKEN") else ""
!python -m src.export.export_model --config configs/train.yaml {push_flag}

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            
New Data Upload               : |          |  0.00B /  0.00B            

  ...l/model/model.safetensors:   0% 550k/540M [00:00<?, ?B/s]

Processing Files (0 / 1)      :   0% 550k/540M [00:01<21:11, 424kB/s,  550kB/s  ]
New Data Upload               :   1% 550k/67.1M [00:01<02:36, 425kB/s,  550kB/s  ]

Processing Files (0 / 1)      :   1% 2.75M/540M [00:01<03:50, 2.34MB/s, 2.29MB/s  ]
New Data Upload               :   2% 2.75M/134M [00:01<00:56, 2.34MB/s, 2.29MB/s  ]

Processing Files (0 / 1)      :   1% 7.15M/540M [00:01<01:23, 6.37MB/s, 5.11MB/s  ]
New Data Upload               :   5% 7.15M/134M [00:01<00:19, 6.37MB/s, 5.11MB/s  ]

Processing Files (0 / 1)      :   3% 17.1M/540M [00:01<00:32, 16.2MB/s, 10.7MB/s  ]
New Data Upload               :  13% 17.1M/134M [00:01<00:07, 16.2MB/s, 10.7MB/s  ]

Processing Files (0 / 1)      :   5% 27.0M/540M [00:02<00:21, 24.3MB/s, 15.0MB/s  ]
New Data Upload               : 

In [10]:
!python -m src.evaluation.manual_tests --config configs/model.yaml

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Manual test rows: 68
Report written to results/manual_test_report.md


Outputs:

- Local artifact: `artifacts/hate_speech_model/`
- Final model: `artifacts/hate_speech_model/model/`
- Hugging Face repo: configured in `configs/train.yaml`